# Working ArXiv Search with Transformers

This notebook combines arXiv data processing with transformer-based semantic search.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import requests
from datetime import datetime
from io import StringIO
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import json
import d6tflow
print("Libraries imported successfully!")

Welcome to d6tflow! For Q&A see https://github.com/d6t/d6tflow
Libraries imported successfully!


In [2]:
# Configuration
query_words = ['machine', 'learning', 'neural', 'network', 'deep']
query_size = 1000  # Start smaller for testing

# Build query URL
queries = [word + '&' for word in query_words]
query = ''.join(queries)
url = f'https://export.arxiv.org/api/query?search_query=all:{query}start=0&max_results={query_size}'
print(f"Query URL: {url}")

search_phrase = ' '.join(query_words[:2])  # "machine learning"
print(f"Search phrase: {search_phrase}")

Query URL: https://export.arxiv.org/api/query?search_query=all:machine&learning&neural&network&deep&start=0&max_results=1000
Search phrase: machine learning


In [3]:
# Fetch data from arXiv
print("Fetching data from arXiv...")
response = requests.get(url)
xml_data = response.text
print(f"Fetched {len(xml_data)} characters of XML data")

Fetching data from arXiv...


Fetched 1961842 characters of XML data


In [4]:
# Parse and process the data
print("Processing arXiv data...")

# Parse XML
df = pd.read_xml(StringIO(xml_data))
print(f"Parsed XML into DataFrame with {len(df)} rows")

# Process data (skip first 7 entries which are usually metadata)
if len(df) > 7:
    papers_df = pd.DataFrame()
    papers_df['title'] = df['title'][7:].reset_index(drop=True)
    papers_df['abstract'] = df['summary'][7:].reset_index(drop=True)
    papers_df['published'] = pd.to_datetime(df['published'][7:].reset_index(drop=True))
    papers_df['updated'] = pd.to_datetime(df['updated'][7:].reset_index(drop=True))
    papers_df['url'] = df['id'][7:].reset_index(drop=True)
    
    # Add analysis columns
    two_years_ago = pd.Timestamp.now(tz='UTC') - pd.DateOffset(years=2)
    papers_df['is_recent'] = papers_df['published'].apply(lambda x: x > two_years_ago)
    papers_df['title_has_keywords'] = papers_df['title'].str.contains(search_phrase, case=False, na=False)
    papers_df['combined_text'] = papers_df['title'] + ' ' + papers_df['abstract']
    
    # Clean up
    papers_df = papers_df.dropna(subset=['title', 'abstract'])
    
    print(f"Processed {len(papers_df)} papers")
    print(f"Recent papers (last 2 years): {papers_df['is_recent'].sum()}")
    print(f"Papers with keywords in title: {papers_df['title_has_keywords'].sum()}")
else:
    print("Warning: Not enough data entries")
    papers_df = pd.DataFrame()

Processing arXiv data...
Parsed XML into DataFrame with 1007 rows
Processed 1000 papers
Recent papers (last 2 years): 128
Papers with keywords in title: 433


In [5]:
# Generate embeddings using transformer model
if len(papers_df) > 0:
    print("Loading transformer model...")
    model = SentenceTransformer('paraphrase-albert-small-v2')
    
    print(f"Generating embeddings for {len(papers_df)} papers...")
    texts = papers_df['combined_text'].tolist()
    embeddings = model.encode(texts, show_progress_bar=True)
    
    print(f"Generated embeddings with shape: {embeddings.shape}")
else:
    print("No papers to process")
    model = None
    embeddings = np.array([])

Loading transformer model...


Generating embeddings for 1000 papers...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Generated embeddings with shape: (1000, 768)


In [6]:
# Define semantic search function
def semantic_search(query, top_k=5):
    """Search for papers similar to the query"""
    if len(papers_df) == 0 or model is None:
        print("No papers available for search")
        return []
    
    # Generate embedding for query
    query_embedding = model.encode([query])
    
    # Calculate similarities
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    # Get top k results
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        result = {
            'title': papers_df.iloc[idx]['title'],
            'abstract': papers_df.iloc[idx]['abstract'],
            'published': papers_df.iloc[idx]['published'],
            'url': papers_df.iloc[idx]['url'],
            'similarity_score': similarities[idx]
        }
        results.append(result)
    
    return results

def display_results(results, max_abstract_length=200):
    """Display search results"""
    for i, paper in enumerate(results, 1):
        print(f"\n{i}. {paper['title']}")
        print(f"   Similarity: {paper['similarity_score']:.3f}")
        print(f"   Published: {str(paper['published'])[:10]}")
        
        abstract = paper['abstract']
        if len(abstract) > max_abstract_length:
            abstract = abstract[:max_abstract_length] + "..."
        print(f"   Abstract: {abstract}")
        print(f"   URL: {paper['url']}")
        print("-" * 80)

print("Search functions defined!")

Search functions defined!


In [7]:
# Example searches
search_queries = [
    "deep learning for natural language processing",
    "computer vision and image recognition",
    "reinforcement learning algorithms"
]

for query in search_queries:
    print(f"\n{'='*60}")
    print(f"SEARCH QUERY: {query}")
    print(f"{'='*60}")
    
    results = semantic_search(query, top_k=3)
    display_results(results)


SEARCH QUERY: deep learning for natural language processing

1. Julia Language in Machine Learning: Algorithms, Applications, and Open
  Issues
   Similarity: 0.561
   Published: 2020-03-23
   Abstract:   Machine learning is driving development across many fields in science and
engineering. A simple and efficient programming language could accelerate
applications of machine learning in various fields...
   URL: http://arxiv.org/abs/2003.10146v2
--------------------------------------------------------------------------------

2. A Mutation-based Text Generation for Adversarial Machine Learning
  Applications
   Similarity: 0.547
   Published: 2022-12-21
   Abstract:   Many natural language related applications involve text generation, created
by humans or machines. While in many of those applications machines support
humans, yet in few others, (e.g. adversarial m...
   URL: http://arxiv.org/abs/2212.11808v1
-------------------------------------------------------------------------------

In [8]:
# Dataset analytics
if len(papers_df) > 0:
    print("📊 Dataset Analytics")
    print("=" * 40)
    print(f"Total papers: {len(papers_df)}")
    print(f"Recent papers (last 2 years): {papers_df['is_recent'].sum()}")
    print(f"Papers with keywords in title: {papers_df['title_has_keywords'].sum()}")
    
    # Publication year distribution
    papers_df['pub_year'] = papers_df['published'].dt.year
    year_counts = papers_df['pub_year'].value_counts().sort_index()
    
    print("\n📅 Publication Year Distribution (recent years):")
    for year in sorted(year_counts.index)[-5:]:  # Last 5 years in data
        count = year_counts[year]
        print(f"  {year}: {count} papers")
    
    # Most common words in titles
    all_titles = ' '.join(papers_df['title'].str.lower())
    words = all_titles.split()
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'}
    filtered_words = [word for word in words if len(word) > 3 and word not in stop_words]
    
    from collections import Counter
    word_counts = Counter(filtered_words)
    
    print("\n🏷️ Most Common Words in Titles:")
    for word, count in word_counts.most_common(10):
        print(f"  {word}: {count}")
else:
    print("No papers available for analytics.")

📊 Dataset Analytics
Total papers: 1000
Recent papers (last 2 years): 128
Papers with keywords in title: 433

📅 Publication Year Distribution (recent years):
  2021: 114 papers
  2022: 76 papers
  2023: 63 papers
  2024: 75 papers
  2025: 31 papers

🏷️ Most Common Words in Titles:
  machine: 656
  learning: 472
  machines: 203
  quantum: 64
  using: 59
  turing: 54
  models: 42
  learning:: 38
  survey: 35
  support: 34


In [9]:
# Export results
if len(papers_df) > 0:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Save papers as CSV
    csv_filename = f"arxiv_papers_{timestamp}.csv"
    papers_df.to_csv(csv_filename, index=False)
    print(f"✅ Saved {len(papers_df)} papers to {csv_filename}")
    
    # Save recent papers as JSON
    recent_papers = papers_df[papers_df['is_recent']]
    if len(recent_papers) > 0:
        json_filename = f"recent_papers_{timestamp}.json"
        recent_papers.to_json(json_filename, orient='records', indent=2, date_format='iso')
        print(f"✅ Saved {len(recent_papers)} recent papers to {json_filename}")
else:
    print("No papers to export.")

✅ Saved 1000 papers to arxiv_papers_20250817_050258.csv


✅ Saved 128 recent papers to recent_papers_20250817_050258.json


## Interactive Search

You can now use the `semantic_search()` function to search for papers. For example:

```python
results = semantic_search("your search query here", top_k=5)
display_results(results)
```

In [10]:
# Custom search - modify this cell to search for what you want
custom_query = "attention mechanisms in transformer models"
print(f"Searching for: {custom_query}")
custom_results = semantic_search(custom_query, top_k=5)
display_results(custom_results)

Searching for: attention mechanisms in transformer models

1. How Much Can We See? A Note on Quantifying Explainability of Machine
  Learning Models
   Similarity: 0.456
   Published: 2019-10-29
   Abstract:   One of the most popular approaches to understanding feature effects of modern
black box machine learning models are partial dependence plots (PDP). These
plots are easy to understand but only able t...
   URL: http://arxiv.org/abs/1910.13376v2
--------------------------------------------------------------------------------

2. Rate-Distortion Theory in Coding for Machines and its Application
   Similarity: 0.452
   Published: 2023-05-26
   Abstract:   Recent years have seen a tremendous growth in both the capability and
popularity of automatic machine analysis of images and video. As a result, a
growing need for efficient compression methods opti...
   URL: http://arxiv.org/abs/2305.17295v2
--------------------------------------------------------------------------------

3. Unmas